In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

# COMMAND ----------

gold_city_activity_path = "abfss://gold@stspmobilitydev001.dfs.core.windows.net/mobility/city_activity"
gold_route_performance_path = "abfss://gold@stspmobilitydev001.dfs.core.windows.net/mobility/route_performance"
gold_mobility_kpis_path = "abfss://gold@stspmobilitydev001.dfs.core.windows.net/dashboard/mobility_kpis"

# COMMAND ----------

city_activity_df = spark.read.format("delta").load(gold_city_activity_path)
route_performance_df = spark.read.format("delta").load(gold_route_performance_path)

# COMMAND ----------

vehicles_per_line_df = (
    route_performance_df.groupBy("event_date", "event_hour")
    .agg(
        F.avg("active_vehicles").alias("avg_vehicles_per_line"),
        F.max("active_vehicles").alias("max_vehicles_in_line"),
        F.sum("active_vehicles").alias("sum_active_vehicles_lines")
    )
)

# COMMAND ----------

mobility_kpis_df = (
    city_activity_df.alias("ca")
    .join(
        vehicles_per_line_df.alias("vl"),
        on=["event_date", "event_hour"],
        how="left"
    )
    .select(
        F.col("ca.event_date"),
        F.col("ca.event_hour"),
        F.col("ca.active_lines"),
        F.col("ca.active_vehicles"),
        F.col("ca.total_positions"),
        F.col("ca.accessible_positions"),
        F.col("ca.accessibility_pct"),
        F.round(F.col("vl.avg_vehicles_per_line"), 2).alias("avg_vehicles_per_line"),
        F.col("vl.max_vehicles_in_line"),
        F.col("vl.sum_active_vehicles_lines")
    )
)

# COMMAND ----------

display(mobility_kpis_df.limit(20))

# COMMAND ----------

mobility_kpis_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save(gold_mobility_kpis_path)

# COMMAND ----------

mobility_kpis_validation_df = spark.read.format("delta").load(gold_mobility_kpis_path)

display(mobility_kpis_validation_df.limit(20))
mobility_kpis_validation_df.printSchema()
print("Total registros validados:", mobility_kpis_validation_df.count())

# COMMAND ----------

display(
    dbutils.fs.ls("abfss://gold@stspmobilitydev001.dfs.core.windows.net/dashboard/mobility_kpis")
)